## N*N矩阵构造
### 采用Dijkstra 算法
非常重要的一点是起点问题

In [1]:
import heapq
import numpy as np
import pandas as pd

#配置路径
graph = {
    1: {2: 3, 3: 4, 8: 1},
    2: {1: 3, 3: 2},
    3: {1: 4, 2: 2, 4: 2, 7: 5, 9: 2},
    4: {3: 2, 9: 1},
    5: {6: 2, 9: 2},
    6: {5: 2, 7: 1, 9: 3},
    7: {3: 5, 6: 1, 8: 2, 9: 4},
    8: {1: 1, 7: 2},
    9: {3: 2, 4: 1, 5: 2, 6: 3, 7: 4}
}

def dijkstra(graph, start_node):
    """
    单源最短路径：计算从 start_node 到图中所有其他节点的最短距离
    """
    # 初始化距离字典，所有节点默认距离为无穷大
    distances = {node: float('infinity') for node in graph}
    distances[start_node] = 0

    # 优先队列 (距离, 节点编号)，用于每次取出当前距离最小的节点
    priority_queue = [(0, start_node)]

    while priority_queue:
        # 取出当前距离最短的节点
        current_distance, current_node = heapq.heappop(priority_queue)

        # 如果取出的距离大于已记录的最短距离，直接跳过（说明是旧数据）
        if current_distance > distances[current_node]:
            continue

        # 遍历相邻节点，更新距离
        for neighbor, weight in graph[current_node].items():
            distance = current_distance + weight

            # 只有找到更短的路径时才更新并入队
            if distance < distances[neighbor]:
                distances[neighbor] = distance
                heapq.heappush(priority_queue, (distance, neighbor))

    return distances

def build_distance_matrix(graph, target_nodes=None):
    """
    构建 n * n 距离矩阵
    如果提供了 target_nodes，则只提取这些目标节点之间的子矩阵
    """
    all_nodes = sorted(list(graph.keys()))
    if target_nodes is None:
        target_nodes = all_nodes

    n = len(target_nodes)
    matrix = np.zeros((n, n))

    # 对每一个目标节点运行一次 Dijkstra
    for i, start in enumerate(target_nodes):
        # 得到从 start 到所有节点的最短距离
        shortest_paths = dijkstra(graph, start)

        # 将结果填入矩阵对应的行中
        for j, end in enumerate(target_nodes):
            matrix[i][j] = shortest_paths[end]

    return matrix, target_nodes

#全图距离矩阵
full_matrix, all_nodes = build_distance_matrix(graph)
print("=== 全图距离矩阵 ===")
df_full = pd.DataFrame(full_matrix, index=all_nodes, columns=all_nodes, dtype=int)
print(df_full)
print("\n")

# ==========================================
# 3. 提取目标景点的 n*n 降维矩阵 (宏观抽象)
# 假设图中标红的节点 (1, 3, 6, 9) 是我们真正要游玩的景点
# ==========================================
#red_nodes = [1, 3, 6, 9]
#target_matrix, t_nodes = build_distance_matrix(graph, target_nodes=red_nodes)
#print("=== 4x4 目标景点抽象矩阵 (仅包含节点 1, 3, 6, 9) ===")
#df_target = pd.DataFrame(target_matrix, index=t_nodes, columns=t_nodes, dtype=int)
#print(df_target)

=== 全图距离矩阵 ===
   1  2  3  4  5  6  7  8  9
1  0  3  4  6  6  4  3  1  6
2  3  0  2  4  6  7  6  4  4
3  4  2  0  2  4  5  5  5  2
4  6  4  2  0  3  4  5  7  1
5  6  6  4  3  0  2  3  5  2
6  4  7  5  4  2  0  1  3  3
7  3  6  5  5  3  1  0  2  4
8  1  4  5  7  5  3  2  0  6
9  6  4  2  1  2  3  4  6  0




## 人群分类打分
首先不同项目有四个维度，然后每类人群有权重矩阵，计算得到分数

刺激类：创极速光轮，抱抱龙冲天赛车，七个小矮人矿山车，雷鸣山漂流，古迹探索营的绳索挑战道

沉浸类：疯狂动物城：热力追踪
加勒比海盗——沉落宝藏之战
翱翔·飞越地平线
小飞侠天空奇遇
小熊维尼历险记

互动类：巴斯光年星际营救
胡迪牛仔嘉年华
太空幸会史迪奇
探险家独木舟
船奇戏水滩

休闲类：幻想曲旋转木马
小飞象
旋转疯蜜罐
弹簧狗团团转
爱丽丝梦游仙境迷宫
探秘海妖复仇号

In [2]:
import numpy as np
import pandas as pd

# 1. 定义项目特征矩阵 P (N x 4)
# 假设我们有 3 个项目，4 个维度：[刺激, 沉浸, 互动, 休闲]
projects_data = [
    [0.9, 0.5, 0.1, 0.1],  # 创极速光轮
    [0.3, 0.9, 0.3, 0.4],  # 飞越地平线
    [0.1, 0.3, 0.8, 0.9]   # 旋转木马
]
project_names = ["创极速光轮", "飞越地平线", "旋转木马"]
P = np.array(projects_data)

# 2. 定义人群偏好权重矩阵 W (4 x 3)
# 4 行代表 4 个维度，3 列分别代表：[普通, 亲子, 情侣]
# 注意：每一列的值加起来最好等于 1
weights_data = [
    [0.25, 0.10, 0.40], # 刺激权重
    [0.25, 0.20, 0.40], # 沉浸权重
    [0.25, 0.40, 0.10], # 互动权重
    [0.25, 0.30, 0.10]  # 休闲权重
]
W = np.array(weights_data)

# 3. 矩阵相乘计算效用得分 S (N x 3)
# 使用 np.dot() 或 @ 运算符进行矩阵乘法
S = P @ W

# 4. 结果格式化输出
columns = ["普通游客得分", "亲子游得分", "情侣游得分"]
df_utility = pd.DataFrame(S, index=project_names, columns=columns)

# 保留两位小数
df_utility = df_utility.round(2)
print("=== 项目绝对效用得分矩阵 S ===")
print(df_utility)

=== 项目绝对效用得分矩阵 S ===
       普通游客得分  亲子游得分  情侣游得分
创极速光轮    0.40   0.26   0.58
飞越地平线    0.48   0.45   0.55
旋转木马     0.52   0.66   0.33


### 重点考虑动态时间

给出活动窗具体时间：巡游与固定地点游？

下面的代码中route为路线

In [ ]:
#评估某条路线
import numpy as np
import pandas as pd


# ==========================================
# 1. 动态排队时间函数 (多峰高斯混合模型 GMM)
# ==========================================
def get_dynamic_queue_time(t_current, base_q, peaks):
    """
    计算特定时刻的排队时间

    参数
    ----
    t_current : float
        当前时刻（以开园时间为 0 起点的分钟数）
    base_q : float
        本底排队时间（平峰时最少排队时间）
    peaks : list[tuple]
        高斯峰参数列表，格式 [(A1, mu1, sigma1), (A2, mu2, sigma2), ...]

    返回
    ----
    q_time : float
        当前时刻的预计排队时间（分钟）
    """
    q_time = float(base_q)

    for A, mu, sigma in peaks:
        if sigma <= 0:
            raise ValueError(f"sigma 必须为正数，当前收到 sigma={sigma}")
        q_time += A * np.exp(-((t_current - mu) ** 2) / (2 * sigma ** 2))

    return max(0.0, q_time)


# ==========================================
# 2. 输入检查
# ==========================================
def validate_inputs(route, distance_matrix, project_info, start_node, end_node=None):
    """
    对输入进行基本校验
    """
    dist = np.array(distance_matrix, dtype=float)

    if dist.ndim != 2 or dist.shape[0] != dist.shape[1]:
        raise ValueError("distance_matrix 必须是方阵")

    n = dist.shape[0]

    if start_node < 0 or start_node >= n:
        raise ValueError(f"start_node={start_node} 超出矩阵索引范围")

    if end_node is not None and (end_node < 0 or end_node >= n):
        raise ValueError(f"end_node={end_node} 超出矩阵索引范围")

    if not isinstance(route, (list, tuple)):
        raise TypeError("route 必须是列表或元组")

    for node in route:
        if node < 0 or node >= n:
            raise ValueError(f"route 中节点 {node} 超出矩阵索引范围")
        if node not in project_info:
            raise ValueError(f"route 中节点 {node} 未在 project_info 中定义")

    return dist


# ==========================================
# 3. 状态转移与时间线推演核心引擎（修正版）
# ==========================================
def evaluate_route_timeline(
    route,
    distance_matrix,
    project_info,
    start_time=0,
    start_node=0,
    end_node=None,
    park_close_time=None,
    T_max=None,
    big_penalty=10000,
    return_to_end=False
):
    """
    推演整条游览路线的时间线，并计算总效用与总惩罚

    参数
    ----
    route : list[int]
        纯“游玩项目节点序列”，不含入口/出口
        例如 [1, 3, 9, 6]
    distance_matrix : 2D array-like
        节点间步行时间矩阵（分钟）
    project_info : dict
        每个项目节点的属性字典
    start_time : float
        出发时刻（以开园为 0）
    start_node : int
        起点节点，例如入口/当前位置
    end_node : int or None
        终点节点，例如出口；若 None 表示不单独计算返回
    park_close_time : float or None
        闭园时间（以开园为 0）；若超出则视为不可行/重罚
    T_max : float or None
        游客本次剩余最大可游玩时间
    big_penalty : float
        违反重要约束时的大惩罚
    return_to_end : bool
        是否在游览完 route 后再走到 end_node

    返回
    ----
    result : dict
        包含总效用、总惩罚、最终得分、时间线日志、可行性等信息
    """
    dist = validate_inputs(route, distance_matrix, project_info, start_node, end_node)

    current_time = float(start_time)
    current_node = start_node

    total_utility = 0.0
    total_penalty = 0.0
    timeline_log = []
    feasible = True
    violation_reasons = []

    # route 为空时，允许直接返回
    if len(route) == 0:
        if return_to_end and end_node is not None and end_node != start_node:
            walk_time = dist[start_node, end_node]
            arrive_time = current_time + walk_time

            penalty = 0.0
            if park_close_time is not None and arrive_time > park_close_time:
                penalty += big_penalty
                feasible = False
                violation_reasons.append("空路线但返回终点时超过闭园时间")

            if T_max is not None and (arrive_time - start_time) > T_max:
                penalty += big_penalty
                feasible = False
                violation_reasons.append("空路线但返回终点时超过总时长约束")

            total_penalty += penalty

            timeline_log.append({
                '从节点': start_node,
                '前往节点': end_node,
                '项目名称': '终点/出口',
                '项目类型': 'end',
                '状态': 'return_end',
                '步行(分)': round(walk_time, 1),
                '到达时刻': round(arrive_time, 1),
                '等待/占位(分)': 0.0,
                '动态排队(分)': 0.0,
                '开始游玩': np.nan,
                '离开时刻': round(arrive_time, 1),
                '本步效用': 0.0,
                '本步惩罚': round(penalty, 1)
            })
            current_time = arrive_time

        return {
            'total_utility': round(total_utility, 2),
            'total_penalty': round(total_penalty, 2),
            'final_score': round(total_utility - total_penalty, 2),
            'end_time': round(current_time, 2),
            'elapsed_time': round(current_time - start_time, 2),
            'feasible': feasible,
            'violation_reasons': violation_reasons,
            'timeline_log': timeline_log
        }

    # 逐个项目推演
    for next_node in route:
        p_info = project_info[next_node]

        walk_time = dist[current_node, next_node]
        arrive_time = current_time + walk_time

        name = p_info.get('name', f'节点{next_node}')
        node_type = p_info.get('type', 'normal')
        play_duration = float(p_info.get('duration', 0))
        utility_score = float(p_info.get('utility', 0))

        wait_time = 0.0
        queue_time = 0.0
        penalty = 0.0
        gained_utility = 0.0
        status = 'completed'

        # ---------- 先检查是否已经晚于闭园 ----------
        if park_close_time is not None and arrive_time > park_close_time:
            penalty += big_penalty
            total_penalty += penalty
            feasible = False
            violation_reasons.append(f"到达 {name} 时已超过闭园时间")

            timeline_log.append({
                '从节点': current_node,
                '前往节点': next_node,
                '项目名称': name,
                '项目类型': node_type,
                '状态': 'arrive_after_close',
                '步行(分)': round(walk_time, 1),
                '到达时刻': round(arrive_time, 1),
                '等待/占位(分)': 0.0,
                '动态排队(分)': 0.0,
                '开始游玩': np.nan,
                '离开时刻': round(arrive_time, 1),
                '本步效用': 0.0,
                '本步惩罚': round(penalty, 1)
            })
            break

        # ---------- 项目时间逻辑 ----------
        # A. 演出类项目：按时间窗处理
        if node_type == 'show' or ('time_window' in p_info):
            if 'time_window' not in p_info:
                raise ValueError(f"演出项目 {name} 缺少 time_window 参数")

            E_start, L_end = p_info['time_window']

            if arrive_time > L_end:
                # 错过演出：不加效用、不消耗游玩时长
                status = 'missed_show'
                penalty += big_penalty
                actual_start_play = np.nan
                leave_time = arrive_time
                play_duration_effective = 0.0
                gained_utility = 0.0
            else:
                # 提前到需要等待占位
                wait_time = max(0.0, E_start - arrive_time)
                actual_start_play = arrive_time + wait_time
                play_duration_effective = play_duration
                leave_time = actual_start_play + play_duration_effective
                gained_utility = utility_score

        # B. 普通项目：按动态排队时间处理
        else:
            if 'base_q' not in p_info or 'peaks' not in p_info:
                raise ValueError(f"普通项目 {name} 缺少 base_q 或 peaks 参数")

            queue_time = get_dynamic_queue_time(
                t_current=arrive_time,
                base_q=p_info['base_q'],
                peaks=p_info['peaks']
            )
            actual_start_play = arrive_time + queue_time
            leave_time = actual_start_play + play_duration
            play_duration_effective = play_duration
            gained_utility = utility_score

        # ---------- 检查闭园约束 ----------
        if park_close_time is not None and leave_time > park_close_time:
            penalty += big_penalty
            feasible = False
            violation_reasons.append(f"{name} 完成时超过闭园时间")

        # ---------- 检查总时长约束 ----------
        if T_max is not None and (leave_time - start_time) > T_max:
            penalty += big_penalty
            feasible = False
            violation_reasons.append(f"{name} 完成后超过总游玩时长上限")

        # ---------- 累计效用与惩罚（修复问题1） ----------
        total_utility += gained_utility
        total_penalty += penalty

        # ---------- 记录日志 ----------
        timeline_log.append({
            '从节点': current_node,
            '前往节点': next_node,
            '项目名称': name,
            '项目类型': node_type,
            '状态': status,
            '步行(分)': round(walk_time, 1),
            '到达时刻': round(arrive_time, 1),
            '等待/占位(分)': round(wait_time, 1),
            '动态排队(分)': round(queue_time, 1),
            '开始游玩': np.nan if pd.isna(actual_start_play) else round(float(actual_start_play), 1),
            '离开时刻': round(float(leave_time), 1),
            '本步效用': round(gained_utility, 1),
            '本步惩罚': round(penalty, 1)
        })

        # ---------- 更新状态 ----------
        current_time = leave_time
        current_node = next_node

        # 若已经明显不可行，可选择提前终止
        # 这里保留后续扩展空间，目前继续推演也可以；若你想严格一些，可以 break
        if not feasible:
            # 这里先不中断，让日志更完整；若你想节省计算可改为 break
            pass

    # 可选：游览结束后返回终点
    if return_to_end and end_node is not None and current_node != end_node:
        walk_time = dist[current_node, end_node]
        arrive_time = current_time + walk_time
        penalty = 0.0

        if park_close_time is not None and arrive_time > park_close_time:
            penalty += big_penalty
            feasible = False
            violation_reasons.append("返回终点时超过闭园时间")

        if T_max is not None and (arrive_time - start_time) > T_max:
            penalty += big_penalty
            feasible = False
            violation_reasons.append("返回终点后超过总游玩时长上限")

        total_penalty += penalty

        timeline_log.append({
            '从节点': current_node,
            '前往节点': end_node,
            '项目名称': '终点/出口',
            '项目类型': 'end',
            '状态': 'return_end',
            '步行(分)': round(walk_time, 1),
            '到达时刻': round(arrive_time, 1),
            '等待/占位(分)': 0.0,
            '动态排队(分)': 0.0,
            '开始游玩': np.nan,
            '离开时刻': round(arrive_time, 1),
            '本步效用': 0.0,
            '本步惩罚': round(penalty, 1)
        })

        current_time = arrive_time

    return {
        'total_utility': round(total_utility, 2),
        'total_penalty': round(total_penalty, 2),
        'final_score': round(total_utility - total_penalty, 2),
        'end_time': round(current_time, 2),
        'elapsed_time': round(current_time - start_time, 2),
        'feasible': feasible,
        'violation_reasons': violation_reasons,
        'timeline_log': timeline_log
    }


# ==========================================
# 4. 模拟测试
# ==========================================
if __name__ == "__main__":
    # 节点说明：
    # 0 = 入口 / 当前所在位置
    # 1 = 创极速光轮
    # 2 = 奇梦幻影秀
    # 3 = 出口（此处为演示方便单独设置）
    dist_mat = [
        [0, 10, 15, 12],
        [10, 0, 8, 10],
        [15, 8, 0, 6],
        [12, 10, 6, 0]
    ]

    projects = {
        1: {
            'name': '创极速光轮',
            'type': 'normal',
            'duration': 5,
            'utility': 8.5,
            'base_q': 15,
            # 两个高峰：开园后 120 分钟、420 分钟
            'peaks': [(60, 120, 30), (50, 420, 40)]
        },
        2: {
            'name': '奇梦幻影秀',
            'type': 'show',
            'duration': 20,
            'utility': 10.0,
            # 开园后 720 分钟开始，730 分钟最晚入场
            'time_window': (720, 730)
        }
    }

    # 现在 route 只表示“项目序列”，不含入口/出口
    test_route = [1, 2]

    result = evaluate_route_timeline(
        route=test_route,
        distance_matrix=dist_mat,
        project_info=projects,
        start_time=700,          # 19:40 开始这段局部路线
        start_node=0,            # 从入口/当前位置出发
        end_node=3,              # 最后返回出口
        park_close_time=750,     # 例如 20:30 闭园
        T_max=80,                # 这段剩余最多可游玩 80 分钟
        big_penalty=10000,
        return_to_end=True
    )

    print("=== 路线评估完成 ===")
    print(f"总效用: {result['total_utility']}")
    print(f"总惩罚: {result['total_penalty']}")
    print(f"最终综合得分(效用-惩罚): {result['final_score']}")
    print(f"结束时刻: {result['end_time']}")
    print(f"总耗时: {result['elapsed_time']}")
    print(f"是否可行: {result['feasible']}")
    print(f"约束违反原因: {result['violation_reasons']}\n")

    print(pd.DataFrame(result['timeline_log']).to_string(index=False))

# 下一步就是搜索最优
三种方案：遗传，退火，蚁群

先说说遗传，大致就是随机生成 100 条不同的路线作为一个“种群“，把得分低的路线淘汰；让得分高的路线之间交换部分基因（比如把路线 A 的前半段和路线 B 的后半段拼接）；偶尔随机打乱某些路线的顺序（变异，防止陷入局部最优）。

然后是退火，从一条随机路线开始，算出它的得分，随机对这条路线做一个小修改（比如交换“创极速光轮”和“飞越地平线”的游玩顺序）。如果修改后得分变高了，直接接受新路线；如果得分变低了，不直接抛弃，而是以一定的概率接受它。这个概率由当前的“温度”决定——初期温度高，愿意频繁接受烂路线（为了跳出局部最优坑）；后期温度低，变得非常挑剔，只接受更好的路线。

最后是蚁群，在起点放出几十只“蚂蚁”，蚂蚁在走到岔路口时，选择下一个项目的概率取决于两点：一是两点之间的物理距离（越近越想去），二是这条路上残留的信息素浓度。每只蚂蚁走完一整条路线后，按照它的总得分，在它走过的路径上留下信息素。得分越高的蚂蚁，留下的信息素越浓。下一代蚂蚁出发时，就会被上一代优秀蚂蚁留下的信息素吸引，最终所有的蚂蚁都会收敛到一条最优路线上。